# Аудит корпуса ambientCG

Категория определяется префиксом имени до первой цифры. Точные совпадения проверяются после приведения к RGB. Расстояние dHash до 4 бит служит фильтром для визуальной проверки и не устанавливает родство материалов. Почти однородные плитки сравниваются только на точное совпадение.

In [1]:
from pathlib import Path
from io import BytesIO
from hashlib import sha256
from itertools import combinations
import json
import re
import numpy as np
import pandas as pd
from PIL import Image

repo = "dream-textures/textures-color-1k"
revision = "33c49b76fcf3a18b0d521d8e760c88d49f3e47bc"
from huggingface_hub import hf_hub_download
import pyarrow.parquet as pq

parquet = hf_hub_download(repo, "data/train-00000-of-00001-5049fdc57d654dc7.parquet",
                          revision=revision, repo_type="dataset")
data = (row for batch in pq.ParquetFile(parquet).iter_batches(batch_size=16)
        for row in batch.to_pylist())
rows = []
rejected = []
for i, row in enumerate(data):
    asset = row["image"]
    mid = Path(asset["path"]).stem
    try:
        im = Image.open(BytesIO(asset["bytes"]))
        im.load()
    except (OSError, ValueError) as error:
        rejected.append(dict(row=i, material_id=mid, reason=type(error).__name__))
        continue
    with im:
        rgb = im.convert("RGB")
        gray = rgb.convert("L")
        lo, hi = np.percentile(np.asarray(gray), [1, 99])
        thumb = np.asarray(gray.resize((9, 8), Image.Resampling.LANCZOS))
        dhash = np.packbits(thumb[:, 1:] > thumb[:, :-1]).tobytes().hex()
        rows.append(dict(row=i, material_id=mid, category=re.split(r"\d", mid, 1)[0],
                         caption=row["text"], width=im.width, height=im.height, mode=im.mode,
                         flat=bool(lo == hi), sha256_rgb=sha256(rgb.tobytes()).hexdigest(), dhash=dhash))
frame = pd.DataFrame(rows)
frame.to_csv("catalog_audit.csv", index=False)
pairs = []
for a, b in combinations(rows, 2):
    exact = (a["width"], a["height"], a["sha256_rgb"]) == (b["width"], b["height"], b["sha256_rgb"])
    distance = (int(a["dhash"], 16) ^ int(b["dhash"], 16)).bit_count()
    if exact or (not a["flat"] and not b["flat"] and distance <= 4):
        pairs.append([a["material_id"], b["material_id"], distance, exact])
pd.DataFrame(pairs, columns=["left", "right", "distance", "exact"]).to_csv("pairs.csv", index=False)
pd.DataFrame(rejected, columns=["row", "material_id", "reason"]).to_csv("rejected.csv", index=False)
summary = dict(rejected=len(rejected), rows=len(rows), categories=int(frame.category.nunique()), flat=int(frame.flat.sum()),
               repeated_ids=int(frame.material_id.duplicated().sum()), modes=frame["mode"].value_counts().to_dict(),
               exact_pairs=sum(p[3] for p in pairs), candidate_pairs=sum(not p[3] for p in pairs),
               categories_by_size=frame.category.value_counts().to_dict())
Path("summary.json").write_text(json.dumps(summary))
print(json.dumps(summary, indent=2))

{
  "rejected": 1,
  "rows": 1425,
  "categories": 77,
  "flat": 11,
  "repeated_ids": 0,
  "modes": {
    "RGB": 1403,
    "L": 22
  },
  "exact_pairs": 2,
  "candidate_pairs": 203,
  "categories_by_size": {
    "PavingStones": 131,
    "Tiles": 113,
    "Bricks": 94,
    "Wood": 90,
    "Fabric": 77,
    "WoodFloor": 60,
    "Ground": 60,
    "Metal": 55,
    "Rock": 53,
    "Leather": 48,
    "Planks": 46,
    "Concrete": 40,
    "Gravel": 35,
    "Marble": 26,
    "Facade": 26,
    "Sign": 26,
    "Asphalt": 25,
    "Rocks": 25,
    "MetalPlates": 20,
    "Terrazzo": 20,
    "DiamondPlate": 20,
    "Wicker": 18,
    "PaintedPlaster": 18,
    "PaintedMetal": 18,
    "Carpet": 15,
    "Granite": 14,
    "WoodSiding": 13,
    "MetalWalkway": 13,
    "Bark": 12,
    "Fence": 12,
    "ChristmasTreeOrnament": 12,
    "Plastic": 11,
    "RoofingTiles": 8,
    "Fingerprints": 8,
    "TactilePaving": 7,
    "Chip": 7,
    "Road": 7,
    "Wallpaper": 6,
    "Snow": 6,
    "Rust": 6,
    "Pla